In [43]:
from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Earthing Grid.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Earthing Grid.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Earthing Grid.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


In [44]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                  X              Y     rotation       start_x        start_y  \
count   7819.000000    7819.000000  7819.000000    980.000000     980.000000   
mean   35045.796376  128133.433497     4.857399  35236.105002  128443.735376   
std      597.053527     961.123281    28.955369    773.939302    1033.335120   
min    33667.550000  125996.966474     0.000000  32641.711101  125582.246706   
25%    34601.690000  127357.990125     0.000000  34620.258125  127753.426275   
50%    34979.686308  128157.844350     0.000000  35354.320100  128750.033800   
75%    35462.831559  128996.163820     0.000000  35842.795400  129234.142325   
max    36625.890000  129809.225013   270.000000  37000.000000  130000.000000   

              end_x          end_y       length  
count    980.000000     980.000000   980.000000  
mean   35262.562492  128468.884823   358.735503  
std      769.312896    1020.494516  1214.176653  
min    33000.000000  125924.176200     0.052880  
25%    34661.737450  127794.9

In [45]:
tracker_mask = (ELM_df['name'] == "EMF_Tracker_1PX64_South") | (ELM_df['name'] == "EMF_Tracker_1PX64_North") | (ELM_df['name'] == "EMF_Tracker_1PX32_North") | (ELM_df['name'] == "EMF_Tracker_1PX32_South")

tracker_df = ELM_df[tracker_mask].copy()

tracker_df.rename(columns={'Y': 'Y_north'}, inplace=True)
tracker_df.drop(columns=['rotation', 'type', 'handle', 'layer', 'QBT', 'DSJ', 'PVM', 'UFV', 'PST', 'INV', 'GRP', 'start_x', 'start_y', 'end_x', 'end_y', 'length'], inplace=True)

tracker_df['Y_south'] = np.where(
    (tracker_df['name'] == "EMF_Tracker_1PX64_South") | (tracker_df['name'] == "EMF_Tracker_1PX64_North"),
    tracker_df['Y_north'] - 85.531,
    tracker_df['Y_north'] - 43.003
)


tolerance = 0.1
tracker_df['Line'] = (tracker_df['X'] / tolerance).round().astype(int)
tracker_df.sort_values(by=['Line', 'Y_north'], ascending=[True, False], inplace=True)
tracker_df.reset_index(drop=True, inplace=True)
print(tracker_df.describe())
print(tracker_df.info())
print(tracker_df)

                  X        Y_north        Y_south           Line
count   7326.000000    7326.000000    7326.000000    7326.000000
mean   35044.698194  128136.006460  128063.740062  350447.124352
std      597.449759     961.921095     963.440178    5974.483337
min    33667.550000  126040.301525  125997.298024  336676.000000
25%    34601.686308  127357.990125  127272.459125  346017.000000
50%    34979.686071  128157.844350  128072.313350  349797.000000
75%    35458.486269  128999.441525  128939.470525  354585.000000
max    36625.890000  129809.225013  129723.694013  366259.000000
<class 'pandas.DataFrame'>
RangeIndex: 7326 entries, 0 to 7325
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   name     7326 non-null   str    
 1   X        7326 non-null   float64
 2   Y_north  7326 non-null   float64
 3   Y_south  7326 non-null   float64
 4   Line     7326 non-null   int64  
dtypes: float64(3), int64(1), str(1)
memory usage: 286

In [46]:
#Different Lines
condition1 = tracker_df["Line"] == tracker_df["Line"].shift()

#Break in the Structure
condition2 = abs(tracker_df["Y_north"] - tracker_df["Y_south"].shift()) < 1.1

connection_mask = condition1 & condition2

print(tracker_df[connection_mask])


                         name             X        Y_north        Y_south  \
2     EMF_Tracker_1PX64_North  33667.553490  128951.820614  128866.289614   
3     EMF_Tracker_1PX64_North  33667.553490  128865.789564  128780.258564   
7     EMF_Tracker_1PX64_North  33673.853490  128865.789564  128780.258564   
10    EMF_Tracker_1PX64_North  33680.153490  128951.820614  128866.289614   
11    EMF_Tracker_1PX64_North  33680.153490  128865.789564  128780.258564   
...                       ...           ...            ...            ...   
7304  EMF_Tracker_1PX64_North  36518.791051  129477.929525  129392.398525   
7306  EMF_Tracker_1PX64_North  36525.091051  129477.929525  129392.398525   
7308  EMF_Tracker_1PX64_North  36531.391051  129477.929525  129392.398525   
7310  EMF_Tracker_1PX32_North  36537.691051  129477.933920  129434.930920   
7312  EMF_Tracker_1PX32_North  36543.991051  129477.933920  129434.930920   

        Line  
2     336676  
3     336676  
7     336739  
10    336802  


In [47]:
tracker_df['Y_south_shifted'] = tracker_df['Y_south'].shift()
tracker_df['Script'] = tracker_df.apply(
    lambda row: f"_.pline {round(row['X'],4)},{round(row['Y_north'],4)} {round(row['X'],4)},{round(row['Y_south_shifted'],4)} ",
    axis=1
)

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Conexão aterramento\PV Paiva - Conexão entre trackers.scr", 'w') as f:
    f.write('\n'.join(tracker_df[connection_mask]["Script"].astype(str)) + ' \n')